# Random Forest — Hyperparameter tuning (Moses)

Starter: `Abhishek_RF.ipynb`. Same data path, sample (`500_000`, `random_state=42`), feature drops, label encoding, and train/test split (`test_size=0.2`, `random_state=42`, stratify on congestion).

This notebook tunes **RandomForestRegressor** and **RandomForestClassifier** using a **validation split inside the training set**, then refits the best settings on **full** `X_train` and reports metrics on the **same** `X_test` as the starter.

In [ ]:
# Imports — match Abhishek_RF starter
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score,
    accuracy_score,
    roc_auc_score,
    f1_score,
)

import matplotlib.pyplot as plt

try:
    plt.style.use("seaborn-v0_8")
except Exception:
    plt.style.use("default")

print("Libraries loaded.")


Libraries loaded.


In [2]:
# Load data + sample (same as Abhishek_RF)
data = pd.read_parquet("../../data/processed/taxi_engineered.parquet")
print(f"Full dataset: {data.shape[0]:,} rows × {data.shape[1]} columns")

SAMPLE_SIZE = 500_000
data = data.sample(n=SAMPLE_SIZE, random_state=42)
print(f"Sampled: {data.shape[0]:,} rows")


Full dataset: 2,451,103 rows × 45 columns
Sampled: 500,000 rows


In [3]:
# Targets + drop leaky / post-trip columns (same list as Abhishek_RF)
y_duration = data["trip_duration_min"]
y_congestion = data["has_congestion_fee"]

drop_cols = [
    "trip_duration_min",
    "has_congestion_fee",
    "tpep_pickup_datetime",
    "fare_amount", "extra", "mta_tax", "tip_amount",
    "tolls_amount", "improvement_surcharge", "total_amount",
    "Airport_fee", "congestion_surcharge", "cbd_congestion_fee",
    "avg_speed_mph",
    "tip_to_total_ratio",
    "is_extreme_fare",
    "cbd_fee_ratio",
    "total_surcharges",
    "surcharges_ratio",
    "base_fare_ratio",
    "store_and_fwd_encoded",
    "VendorID",
    "payment_type",
    "payment_name",
]

existing_drops = [c for c in drop_cols if c in data.columns]
X = data.drop(columns=existing_drops)
print(f"Dropped {len(existing_drops)} columns → {X.shape[1]} features")


Dropped 24 columns → 21 features


In [4]:
# Label-encode categoricals + train/test split (same as starter)
cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()

label_encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    if X[col].dtype.name == "category":
        X[col] = X[col].astype(str)
    X[col] = X[col].fillna("Unknown")
    X[col] = le.fit_transform(X[col])
    label_encoders[col] = le

feature_names = X.columns.tolist()

X_train, X_test, y_dur_train, y_dur_test, y_cong_train, y_cong_test = train_test_split(
    X, y_duration, y_congestion,
    test_size=0.2,
    random_state=42,
    stratify=y_congestion,
)

print(f"Train: {X_train.shape[0]:,} | Test: {X_test.shape[0]:,}")


Train: 400,000 | Test: 100,000


## Tuning strategy

1. Split training again: 80% `X_tr` / 20% `X_val` (`random_state=42`, stratify congestion).
2. Fit each candidate on `X_tr`, score on `X_val` (MAE for duration in minutes; ROC-AUC for congestion).
3. Refit the winner on all `X_train`, evaluate on `X_test` (same holdout as Abhishek).



In [5]:
# Inner split for hyperparameter search
X_tr, X_val, y_dur_tr, y_dur_val, y_cong_tr, y_cong_val = train_test_split(
    X_train, y_dur_train, y_cong_train,
    test_size=0.2,
    random_state=42,
    stratify=y_cong_train,
)
print(f"Tuning fit rows: {X_tr.shape[0]:,} | Tuning validation: {X_val.shape[0]:,}")


Tuning fit rows: 320,000 | Tuning validation: 80,000


In [6]:
# Regression: candidate settings (tune this list)
REG_CANDIDATES = [
    {"name": "starter_match", "n_estimators": 100, "max_depth": 30, "min_samples_leaf": 5},
    {"name": "more_trees_sqrt", "n_estimators": 300, "max_depth": 25, "min_samples_leaf": 5, "max_features": "sqrt"},
    {"name": "more_trees_shallow", "n_estimators": 400, "max_depth": 20, "min_samples_leaf": 10, "max_features": "sqrt"},
    {"name": "deep_many_trees", "n_estimators": 500, "max_depth": 30, "min_samples_leaf": 5, "max_features": "sqrt"},
    {"name": "regularized_leaf", "n_estimators": 300, "max_depth": 25, "min_samples_leaf": 15, "max_features": "sqrt"},
    {"name": "split_reg", "n_estimators": 400, "max_depth": 25, "min_samples_leaf": 5, "min_samples_split": 10, "max_features": "sqrt"},
    {"name": "half_features", "n_estimators": 400, "max_depth": 25, "min_samples_leaf": 5, "max_features": 0.5},
    {"name": "unlimited_depth", "n_estimators": 350, "max_depth": None, "min_samples_leaf": 8, "max_features": "sqrt"},
]


def eval_reg_config(cfg):
    cfg = dict(cfg)
    name = cfg.pop("name")
    model = RandomForestRegressor(random_state=42, n_jobs=-1, **cfg)
    model.fit(X_tr, y_dur_tr)
    pred = model.predict(X_val)
    mae = mean_absolute_error(y_dur_val, pred)
    rmse = np.sqrt(mean_squared_error(y_dur_val, pred))
    r2 = r2_score(y_dur_val, pred)
    return {"name": name, **cfg, "val_MAE": mae, "val_RMSE": rmse, "val_R2": r2}


rows = [eval_reg_config(c) for c in REG_CANDIDATES]
reg_search = pd.DataFrame(rows).sort_values("val_MAE")
print("Validation leaderboard (MAE on tuning validation, lower is better):")
print(reg_search.to_string(index=False))

best = reg_search.iloc[0]
best_reg_name = best["name"]
best_reg_params = {"n_estimators": int(best["n_estimators"]), "min_samples_leaf": int(best["min_samples_leaf"])}
md = best["max_depth"]
if pd.isna(md):
    best_reg_params["max_depth"] = None
else:
    best_reg_params["max_depth"] = None if md is None else int(md)
if "min_samples_split" in best.index and pd.notna(best["min_samples_split"]):
    best_reg_params["min_samples_split"] = int(best["min_samples_split"])
if "max_features" in best.index and pd.notna(best["max_features"]):
    best_reg_params["max_features"] = best["max_features"]

print(f"\nSelected: {best_reg_name} -> {best_reg_params}")


Validation leaderboard (MAE on tuning validation, lower is better):
              name  n_estimators  max_depth  min_samples_leaf  val_MAE  val_RMSE   val_R2 max_features  min_samples_split
     starter_match           100       30.0                 5 2.645879  4.086806 0.853767          NaN                NaN
     half_features           400       25.0                 5 2.655089  4.102969 0.852608          0.5                NaN
   deep_many_trees           500       30.0                 5 2.720741  4.180769 0.846966         sqrt                NaN
         split_reg           400       25.0                 5 2.724212  4.184423 0.846698         sqrt               10.0
   more_trees_sqrt           300       25.0                 5 2.725249  4.186142 0.846572         sqrt                NaN
   unlimited_depth           350        NaN                 8 2.745491  4.215550 0.844409         sqrt                NaN
more_trees_shallow           400       20.0                10 2.773137  4.2506

In [ ]:
# Final regressor: full X_train, same X_test as starter
rf_reg_tuned = RandomForestRegressor(random_state=42, n_jobs=-1, **best_reg_params)
rf_reg_tuned.fit(X_train, y_dur_train)

y_dur_pred_test = rf_reg_tuned.predict(X_test)
test_rmse = float(np.sqrt(mean_squared_error(y_dur_test, y_dur_pred_test)))
test_mae = float(mean_absolute_error(y_dur_test, y_dur_pred_test))
test_r2 = float(r2_score(y_dur_test, y_dur_pred_test))

STARTER_TEST_RMSE = 3.99
STARTER_TEST_MAE = 2.61
STARTER_TEST_R2 = 0.8579

print("=" * 60)
print("TUNED RF REGRESSOR — TEST SET")
print("=" * 60)
print(f"Test RMSE: {test_rmse:.4f}  (starter: {STARTER_TEST_RMSE:.2f})")
print(f"Test MAE:  {test_mae:.4f}   (starter: {STARTER_TEST_MAE:.2f})")
print(f"Test R2:   {test_r2:.4f}   (starter: {STARTER_TEST_R2:.4f})")
print(f"\nDelta RMSE (starter − tuned): {STARTER_TEST_RMSE - test_rmse:+.4f} min  (positive => tuned better)")
print(f"Delta MAE  (starter − tuned): {STARTER_TEST_MAE - test_mae:+.4f} min")


TUNED RF REGRESSOR — TEST SET
Test RMSE: 3.9908  (starter: 3.99)
Test MAE:  2.6071   (starter: 2.61)
Test R2:   0.8579   (starter: 0.8579)

Delta RMSE (starter − tuned): -0.0008 min  (positive => tuned better)
Delta MAE  (starter − tuned): +0.0029 min


### Optional: `log1p(duration)` target

Feedback suggested log-scaling duration. Trains on `log1p(y)` and evaluates MAE/RMSE on **minutes** via `expm1`.

In [ ]:
# Log target (same hyperparameters as tuned level-target model)
y_dur_train_log = np.log1p(y_dur_train)

rf_log = RandomForestRegressor(random_state=42, n_jobs=-1, **best_reg_params)
rf_log.fit(X_train, y_dur_train_log)
pred_minutes = np.expm1(rf_log.predict(X_test))

test_mae_log = mean_absolute_error(y_dur_test, pred_minutes)
test_rmse_log = np.sqrt(mean_squared_error(y_dur_test, pred_minutes))
test_r2_log = r2_score(y_dur_test, pred_minutes)

print("Log-target RF — test (back to minutes):")
print(f"  RMSE: {test_rmse_log:.4f}  |  MAE: {test_mae_log:.4f}  |  R2: {test_r2_log:.4f}")
print(f"  Level-target tuned test MAE was: {test_mae:.4f}")


Log-target RF — test (back to minutes):
  RMSE: 3.9939  |  MAE: 2.5699  |  R2: 0.8577
  Level-target tuned test MAE was: 2.6071


In [9]:
# Classification candidates
CLF_CANDIDATES = [
    {"name": "starter_match", "n_estimators": 100, "max_depth": 30, "min_samples_leaf": 5, "class_weight": "balanced"},
    {"name": "more_trees_sqrt", "n_estimators": 300, "max_depth": 25, "min_samples_leaf": 5, "max_features": "sqrt", "class_weight": "balanced"},
    {"name": "regularized", "n_estimators": 400, "max_depth": 20, "min_samples_leaf": 10, "max_features": "sqrt", "class_weight": "balanced"},
    {"name": "deep", "n_estimators": 500, "max_depth": 30, "min_samples_leaf": 5, "max_features": "sqrt", "class_weight": "balanced"},
    {"name": "half_feat", "n_estimators": 400, "max_depth": 25, "min_samples_leaf": 5, "max_features": 0.5, "class_weight": "balanced"},
]


def eval_clf_config(cfg):
    cfg = dict(cfg)
    name = cfg.pop("name")
    model = RandomForestClassifier(random_state=42, n_jobs=-1, **cfg)
    model.fit(X_tr, y_cong_tr)
    proba = model.predict_proba(X_val)[:, 1]
    pred = model.predict(X_val)
    return {
        "name": name,
        **cfg,
        "val_accuracy": accuracy_score(y_cong_val, pred),
        "val_roc_auc": roc_auc_score(y_cong_val, proba),
        "val_f1": f1_score(y_cong_val, pred),
    }


clf_rows = [eval_clf_config(c) for c in CLF_CANDIDATES]
clf_search = pd.DataFrame(clf_rows).sort_values("val_roc_auc", ascending=False)
print("Classifier validation leaderboard (ROC-AUC):")
print(clf_search.to_string(index=False))

best_c = clf_search.iloc[0]
cmd = best_c["max_depth"]
best_clf_params = {
    "n_estimators": int(best_c["n_estimators"]),
    "max_depth": None if pd.isna(cmd) else (None if cmd is None else int(cmd)),
    "min_samples_leaf": int(best_c["min_samples_leaf"]),
    "class_weight": best_c["class_weight"],
}
if "max_features" in best_c.index and pd.notna(best_c["max_features"]):
    best_clf_params["max_features"] = best_c["max_features"]

print(f"\nSelected: {best_c['name']} -> {best_clf_params}")


Classifier validation leaderboard (ROC-AUC):
           name  n_estimators  max_depth  min_samples_leaf class_weight  val_accuracy  val_roc_auc   val_f1 max_features
      half_feat           400         25                 5     balanced      0.962175     0.984958 0.974305          0.5
           deep           500         30                 5     balanced      0.951650     0.978493 0.967249         sqrt
more_trees_sqrt           300         25                 5     balanced      0.951738     0.978401 0.967310         sqrt
  starter_match           100         30                 5     balanced      0.951800     0.978253 0.967340          NaN
    regularized           400         20                10     balanced      0.947087     0.976792 0.964104         sqrt

Selected: half_feat -> {'n_estimators': 400, 'max_depth': 25, 'min_samples_leaf': 5, 'class_weight': 'balanced', 'max_features': 0.5}


In [ ]:
# Final classifier: full train, test set
rf_clf_tuned = RandomForestClassifier(random_state=42, n_jobs=-1, **best_clf_params)
rf_clf_tuned.fit(X_train, y_cong_train)

y_cong_pred_test = rf_clf_tuned.predict(X_test)
y_cong_proba_test = rf_clf_tuned.predict_proba(X_test)[:, 1]

print("=" * 60)
print("TUNED RF CLASSIFIER — TEST SET")
print("=" * 60)
print(f"Accuracy: {accuracy_score(y_cong_test, y_cong_pred_test):.4f}")
print(f"ROC-AUC:  {roc_auc_score(y_cong_test, y_cong_proba_test):.4f}")
print(f"F1:       {f1_score(y_cong_test, y_cong_pred_test):.4f}")


TUNED RF CLASSIFIER — TEST SET
Accuracy: 0.9636
ROC-AUC:  0.9855
F1:       0.9753


In [11]:
# Top feature importances — tuned regressor
imp = (
    pd.Series(rf_reg_tuned.feature_importances_, index=feature_names)
    .sort_values(ascending=False)
    .head(15)
)
print("Top 15 features (tuned regressor):")
print(imp.to_string())


Top 15 features (tuned regressor):
trip_distance         0.843819
pickup_hour           0.029798
DOLocationID          0.028636
time_of_day           0.020923
PULocationID          0.020682
time_slot             0.017311
hour_x_dayofweek      0.009816
dropoff_borough       0.005844
is_rush_hour          0.005016
pickup_day_of_week    0.004249
is_airport_trip       0.003093
pickup_borough        0.002814
is_same_borough       0.001588
ratecode_name         0.001582
passenger_count       0.001450
